# Week 4: Ablation Study (N=1000, auto-save)
**Project**: Quantum Kernels on CLIP/BERT Embeddings

**Every kernel matrix is saved + downloaded the instant it finishes** — so if the
session dies after one 83-min ablation, you keep it and only redo what was running.
Every figure downloads as PNG + PDF.

**Ablations:**
1. Feature maps — Angle vs IQP 1L vs IQP 2L (8 qubits)
2. Qubit count — 4 vs 8 (IQP 1L)
3. C parameter — sweep at N=200, apply best at N=1000
4. Multi-seed — 5 seeds (optional, ~7 hrs)

**Inputs:** `embeddings_week2.npz`, `baseline_results.pkl`, `kernel_matrices_week3.npz`

**Runtime:** ~4.2 hrs without multi-seed (3 fresh kernel matrices at ~83 min each)

In [ ]:
!pip install pennylane pennylane-lightning --quiet
import pennylane as qml
import numpy as np
import time, json, pickle, os, shutil, warnings
import matplotlib.pyplot as plt
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, f1_score
warnings.filterwarnings('ignore')
print(f"PennyLane {qml.__version__}")
dev4 = qml.device('lightning.qubit', wires=4)
dev8 = qml.device('lightning.qubit', wires=8)

---
## Step 0: Storage + checkpoint helpers (same as Week 3)

In [ ]:
USE_DRIVE = True
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    SAVE_DIR = '/content/drive/MyDrive/qml_project'
else:
    SAVE_DIR = 'qml_project'
os.makedirs(SAVE_DIR, exist_ok=True)
print(f"Save directory: {SAVE_DIR}")

def _persist_and_download(path):
    if os.path.abspath(SAVE_DIR) != os.getcwd():
        shutil.copy(path, os.path.join(SAVE_DIR, os.path.basename(path)))
        print(f"  saved -> {SAVE_DIR}/{os.path.basename(path)}")
    try:
        from google.colab import files
        files.download(path)
        print(f"  download -> {os.path.basename(path)}")
    except Exception as e:
        print(f"  (download skipped: {type(e).__name__} — file is saved above)")

def checkpoint(filename, save_fn):
    save_fn(filename); _persist_and_download(filename)

def save_figure(fig, name):
    for ext in ('png', 'pdf'):
        p = f"{name}.{ext}"
        fig.savefig(p, bbox_inches='tight', dpi=150)
        _persist_and_download(p)

def _find(fname):
    dp = os.path.join(SAVE_DIR, fname)
    return dp if os.path.exists(dp) else fname

print("Helpers ready.")

---
## Step 1: Load data + Week 3 kernel matrix

In [ ]:
data = np.load(_find('embeddings_week2.npz'))
wk3  = np.load(_find('kernel_matrices_week3.npz'))

X_tr_full = data['X_tr_subset'].astype(np.float64)
X_te_full = data['X_te_subset'].astype(np.float64)
y_tr_full = data['y_tr_subset']
y_te_full = data['y_te_subset']

K_wk3_train = wk3['K_train_quantum']   # 1000x1000 IQP 1L
K_wk3_test  = wk3['K_test_quantum']    # 200x1000

with open(_find('baseline_results.pkl'), 'rb') as f:
    baselines = pickle.load(f)

N_ABL = 1000
X_tr, y_tr = X_tr_full[:N_ABL], y_tr_full[:N_ABL]
X_te, y_te = X_te_full, y_te_full

print(f"Train {X_tr.shape}  Test {X_te.shape}")
print(f"Week 3 kernel: train {K_wk3_train.shape}, test {K_wk3_test.shape}")
assert K_wk3_train.shape == (1000, 1000), "Week 3 matrix not N=1000 — re-run Week 3"
linear_acc = baselines.get('Linear', {}).get('accuracy', 0.67)
print(f"\nBaselines:")
for k, v in baselines.items():
    print(f"  {k:<12} acc={v['accuracy']:.4f}  f1={v['f1']:.4f}")

---
## Step 2: Feature maps

In [ ]:
def angle_map(x, wires):
    for i, w in enumerate(wires): qml.RY(x[i], wires=w)

def iqp_1layer(x, wires):
    n = len(wires)
    for w in wires: qml.Hadamard(wires=w)
    for i, w in enumerate(wires): qml.RZ(x[i], wires=w)
    for i in range(n-1):
        qml.CNOT(wires=[wires[i], wires[i+1]])
        qml.RZ(x[i]*x[i+1], wires=wires[i+1])
        qml.CNOT(wires=[wires[i], wires[i+1]])

def iqp_2layer(x, wires):
    for _ in range(2): iqp_1layer(x, wires)

def make_kernel(fmap, device):
    @qml.qnode(device)
    def circ(x1, x2):
        fmap(x1, wires=range(device.num_wires))
        qml.adjoint(fmap)(x2, wires=range(device.num_wires))
        return qml.probs(wires=range(device.num_wires))
    return lambda a, b: float(circ(a, b)[0])

k_angle   = make_kernel(angle_map,  dev8)
k_iqp1    = make_kernel(iqp_1layer, dev8)
k_iqp2    = make_kernel(iqp_2layer, dev8)
k_iqp1_4q = make_kernel(iqp_1layer, dev4)

print("Sanity (K(x,x) must be 1.0):")
for nm, kf, x in [('Angle',k_angle,X_tr[0]),('IQP1L',k_iqp1,X_tr[0]),
                  ('IQP2L',k_iqp2,X_tr[0]),('IQP1L-4q',k_iqp1_4q,X_tr[0,:4])]:
    print(f"  {nm:<10} {kf(x,x):.6f}")

---
## Ablation 1: Feature maps (8 qubits, N=1000)
**Each new kernel matrix is saved the moment it finishes.**
IQP 1L is reused from Week 3 (free).

In [ ]:
fm_results = {}

# IQP 1L — reuse Week 3
y_w3 = SVC(kernel='precomputed', C=1.0).fit(K_wk3_train, y_tr).predict(K_wk3_test)
fm_results['IQP 1L'] = {'accuracy': float(accuracy_score(y_te, y_w3)),
                        'f1': float(f1_score(y_te, y_w3)), 'reused': True}
print(f"IQP 1L (reused): acc={fm_results['IQP 1L']['accuracy']:.4f}  f1={fm_results['IQP 1L']['f1']:.4f}")

# Angle and IQP 2L — compute + save each immediately
for fname, kfn, tag in [('Angle', k_angle, 'angle'), ('IQP 2L', k_iqp2, 'iqp2l')]:
    print(f"\nComputing {fname} (N={N_ABL}, ~83 min)...")
    t0 = time.time()
    Ktr = qml.kernels.kernel_matrix(X_tr, X_tr, kfn)
    Kte = qml.kernels.kernel_matrix(X_te, X_tr, kfn)
    print(f"  done {((time.time()-t0)/60):.1f} min — saving immediately:")
    checkpoint(f'kernel_{tag}_n1000.npz',
               lambda p, a=Ktr, b=Kte: np.savez(p, K_train=a, K_test=b))
    y_p = SVC(kernel='precomputed', C=1.0).fit(Ktr, y_tr).predict(Kte)
    fm_results[fname] = {'accuracy': float(accuracy_score(y_te, y_p)),
                         'f1': float(f1_score(y_te, y_p))}
    print(f"  {fname}: acc={fm_results[fname]['accuracy']:.4f}  f1={fm_results[fname]['f1']:.4f}")

print(f"\n{'Feature Map':<12}{'Acc':>9}{'F1':>9}")
print("-"*30)
for n, r in fm_results.items():
    print(f"{n:<12}{r['accuracy']:>9.4f}{r['f1']:>9.4f}")

---
## Ablation 2: Qubit count (IQP 1L, N=1000)
4 qubits uses first 4 PCA dims. 8 reused. 12 needs a Week 2 re-run.

In [ ]:
qubit_results = {8: {'accuracy': fm_results['IQP 1L']['accuracy'],
                     'f1': fm_results['IQP 1L']['f1'], 'reused': True}}

print(f"Computing IQP 1L 4-qubit kernel (N={N_ABL}, ~83 min)...")
t0 = time.time()
K4_tr = qml.kernels.kernel_matrix(X_tr[:, :4], X_tr[:, :4], k_iqp1_4q)
K4_te = qml.kernels.kernel_matrix(X_te[:, :4], X_tr[:, :4], k_iqp1_4q)
print(f"  done {((time.time()-t0)/60):.1f} min — saving:")
checkpoint('kernel_iqp1l_4q_n1000.npz',
           lambda p: np.savez(p, K_train=K4_tr, K_test=K4_te))
y_p4 = SVC(kernel='precomputed', C=1.0).fit(K4_tr, y_tr).predict(K4_te)
qubit_results[4] = {'accuracy': float(accuracy_score(y_te, y_p4)),
                    'f1': float(f1_score(y_te, y_p4))}
print(f"  4-qubit: acc={qubit_results[4]['accuracy']:.4f}  f1={qubit_results[4]['f1']:.4f}")

print(f"\n{'Qubits':>7}{'Acc':>9}{'F1':>9}")
for n_q in sorted(qubit_results):
    r = qubit_results[n_q]
    print(f"{n_q:>7}{r['accuracy']:>9.4f}{r['f1']:>9.4f}")
print(f"{'12':>7}   (re-run Week 2 with n_components=12)")

---
## Ablation 3: C parameter (sweep at N=200, apply best at N=1000)

In [ ]:
Kc_tr = qml.kernels.kernel_matrix(X_tr[:200], X_tr[:200], k_iqp1)
Kc_te = qml.kernels.kernel_matrix(X_te, X_tr[:200], k_iqp1)
c_results = {}
print(f"{'C':>6}{'Acc':>9}{'F1':>9}")
print("-"*24)
for C in [0.1, 0.5, 1.0, 5.0, 10.0]:
    yp = SVC(kernel='precomputed', C=C).fit(Kc_tr, y_tr[:200]).predict(Kc_te)
    c_results[C] = {'accuracy': float(accuracy_score(y_te, yp)),
                    'f1': float(f1_score(y_te, yp))}
    print(f"{C:>6.1f}{c_results[C]['accuracy']:>9.4f}{c_results[C]['f1']:>9.4f}")
best_C = max(c_results, key=lambda c: c_results[c]['accuracy'])
print(f"\nBest C = {best_C}")

if best_C == 1.0:
    print("C=1.0 already used everywhere — no re-run needed.")
else:
    yb = SVC(kernel='precomputed', C=best_C).fit(K_wk3_train, y_tr).predict(K_wk3_test)
    print(f"IQP 1L @ C={best_C}, N=1000: acc={accuracy_score(y_te,yb):.4f}  f1={f1_score(y_te,yb):.4f}")

---
## Ablation 4: Multi-seed (optional, ~7 hrs)
Set `RUN_MULTISEED = True`. Each seed's kernel matrix is saved as it completes.

In [ ]:
RUN_MULTISEED = False
seed_accs, seed_f1s = [], []
if RUN_MULTISEED:
    for seed in range(5):
        rng = np.random.RandomState(seed)
        idx = rng.choice(len(X_tr_full), N_ABL, replace=False)
        Xs, ys = X_tr_full[idx], y_tr_full[idx]
        print(f"seed {seed} (~83 min)...")
        t0 = time.time()
        Ks_tr = qml.kernels.kernel_matrix(Xs, Xs, k_iqp1)
        Ks_te = qml.kernels.kernel_matrix(X_te, Xs, k_iqp1)
        checkpoint(f'kernel_seed{seed}_n1000.npz',
                   lambda p, a=Ks_tr, b=Ks_te: np.savez(p, K_train=a, K_test=b))
        yp = SVC(kernel='precomputed', C=best_C).fit(Ks_tr, ys).predict(Ks_te)
        seed_accs.append(accuracy_score(y_te, yp)); seed_f1s.append(f1_score(y_te, yp))
        print(f"  acc={seed_accs[-1]:.4f}  ({(time.time()-t0)/60:.0f} min)")
    print(f"\nMean acc {np.mean(seed_accs):.4f} ± {np.std(seed_accs):.4f}")
else:
    print("Skipped (RUN_MULTISEED=False).")

---
## Paper Table 1

In [ ]:
print("="*70)
print(f"{'Method':<26}{'Qubits':>7}{'N':>6}{'Acc':>8}{'F1':>8}{'vsLin':>9}")
print("-"*70)
for name, v in baselines.items():
    print(f"  {'Classical — '+name:<24}{'—':>7}{'1000':>6}{v['accuracy']:>8.4f}{v['f1']:>8.4f}{v['accuracy']-linear_acc:>+9.1%}")
print()
for fname, r in fm_results.items():
    note = ' *' if fname=='IQP 1L' else ''
    print(f"  {'Quantum — '+fname:<24}{'8':>7}{'1000':>6}{r['accuracy']:>8.4f}{r['f1']:>8.4f}{r['accuracy']-linear_acc:>+9.1%}{note}")
print()
for n_q in sorted(qubit_results):
    r = qubit_results[n_q]
    print(f"  {'Quantum — IQP 1L':<24}{n_q:>7}{'1000':>6}{r['accuracy']:>8.4f}{r['f1']:>8.4f}{r['accuracy']-linear_acc:>+9.1%}")
print(f"  {'Quantum — IQP 1L':<24}{'12':>7}{'—':>6}{'TBD':>8}{'TBD':>8}")
if seed_accs:
    print(f"\n  {'Quantum — IQP1L 5-seed':<24}{'8':>7}{'1000':>6}{np.mean(seed_accs):>8.4f}{np.mean(seed_f1s):>8.4f}  ±{np.std(seed_accs):.4f}")
print("="*70)
print("* reused from Week 3")

---
## Figures (PNG + PDF)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

fm_names = list(fm_results.keys())
fm_accs  = [fm_results[n]['accuracy'] for n in fm_names]
axes[0].bar(fm_names, fm_accs, color=['#4D9DE0','#E15554','#3BB273'], width=0.5, edgecolor='white')
axes[0].axhline(linear_acc, color='gray', ls='--', alpha=0.6, label='Linear baseline')
axes[0].set_ylim(0.5,0.85); axes[0].set_title(f'Feature maps (8q, N={N_ABL})'); axes[0].set_ylabel('Accuracy'); axes[0].legend(fontsize=9)
for b,v in zip(axes[0].patches, fm_accs): axes[0].text(b.get_x()+b.get_width()/2, v+0.004, f'{v:.3f}', ha='center', fontsize=10, fontweight='500')

q_labels = [f'{n}q' for n in sorted(qubit_results)]+['12q*']
q_accs   = [qubit_results[n]['accuracy'] for n in sorted(qubit_results)]+[None]
bars2 = axes[1].bar(q_labels, [v if v else 0 for v in q_accs], color=['#4D9DE0','#E15554','#CCCCCC'], width=0.5, edgecolor='white')
axes[1].axhline(linear_acc, color='gray', ls='--', alpha=0.6, label='Linear baseline')
axes[1].set_ylim(0.5,0.85); axes[1].set_title(f'Qubit count (IQP1L, N={N_ABL})'); axes[1].set_ylabel('Accuracy'); axes[1].legend(fontsize=9)
for b,v in zip(bars2, q_accs):
    axes[1].text(b.get_x()+b.get_width()/2, b.get_height()+0.004, f'{v:.3f}' if v else 'TBD', ha='center', fontsize=10, fontweight='500')

C_vals = list(c_results.keys())
c_accs = [c_results[c]['accuracy'] for c in C_vals]
bi = c_accs.index(max(c_accs))
axes[2].bar([str(c) for c in C_vals], c_accs, color=['#E15554' if i==bi else '#4D9DE0' for i in range(len(C_vals))], width=0.6, edgecolor='white')
axes[2].axhline(linear_acc, color='gray', ls='--', alpha=0.6, label='Linear baseline')
axes[2].set_ylim(0.5,0.85); axes[2].set_title('C sweep (N=200)'); axes[2].set_xlabel('C'); axes[2].set_ylabel('Accuracy'); axes[2].legend(fontsize=9)
for b,v in zip(axes[2].patches, c_accs): axes[2].text(b.get_x()+b.get_width()/2, v+0.004, f'{v:.3f}', ha='center', fontsize=10, fontweight='500')

plt.suptitle(f'Week 4 Ablations — N={N_ABL}, SST-2, 8-dim BERT', y=1.02)
plt.tight_layout(); plt.show()
save_figure(fig, 'fig4_ablations')

---
## Save results JSON

In [ ]:
all_results = {
    'N_ABL': N_ABL,
    'feature_map_ablation': fm_results,
    'qubit_ablation': {str(k): v for k, v in qubit_results.items()},
    'c_sweep': {str(k): v for k, v in c_results.items()},
    'best_C': float(best_C),
    'multi_seed': {'ran': bool(seed_accs),
                   'mean_accuracy': float(np.mean(seed_accs)) if seed_accs else None,
                   'std_accuracy': float(np.std(seed_accs)) if seed_accs else None},
    'classical_baselines': baselines,
}
checkpoint('results_week4.json', lambda p: json.dump(all_results, open(p,'w'), indent=2))
print(f"\nAll files in {SAVE_DIR}:")
for f in sorted(os.listdir(SAVE_DIR)): print(f"  {f}")

---
## Week 4 complete

### Saved + downloaded:
- `kernel_angle_n1000.npz`, `kernel_iqp2l_n1000.npz`, `kernel_iqp1l_4q_n1000.npz` — each saved the instant it finished
- `fig4_ablations.png` / `.pdf`
- `results_week4.json`

### If the session dies mid-ablation:
The completed kernel matrices are in Drive. Next run, load them instead of recomputing:
```python
ka = np.load('/content/drive/MyDrive/qml_project/kernel_angle_n1000.npz')
K_train, K_test = ka['K_train'], ka['K_test']
```

### Week 5:
Analysis — kernel alignment (KTA), t-SNE of the quantum feature space, learning-curve
significance. All inputs are now saved in Drive.